# GuideRail walkthrough (executed)

A citation-first RAG assistant over healthcare guideline documents: it answers only from retrieved text, checks each answer against that evidence, and returns
`needs_review` when support is weak (threshold 0.6).

This notebook has **no logic of its own**. It calls the package, runs the project's scripts, and displays the saved evaluation reports, so the numbers match the README and
`docs/EVALUATION_REPORT.md`. It uses the offline extractive generator (no network, no API key). Outputs are saved in the file, so it can be read without running anything.

**Read the last section before trusting any number:** results on the owner's independent questions are much weaker than the development results.
Research prototype; not for clinical use. The synthetic guideline text has not been clinically reviewed.

In [1]:
import sys, os, json, subprocess
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "evaluation")]
os.chdir(ROOT)
print("project root:", ROOT.name)

project root: guiderail


## 1. Deterministic indexing

The pipeline builds the FAISS index twice, in separate processes, from the synthetic Markdown guidelines and compares SHA-256 hashes.

In [2]:
r = subprocess.run([sys.executable, "scripts/pipeline.py", "--check-determinism"], capture_output=True, text=True, encoding="utf-8")
print(r.stdout.strip()); print(r.stderr.strip()[-300:])

Indexed 22 documents (2 superseded) into 79 chunks -> /Users/amitmukherjee/Developer/guiderail/index
index_sha256 = 45c3d9422161bc9750864850b8285b1feced930110530c2741ec0b9dd986cef8
Determinism check passed: two independent builds produced identical hashes.



In [3]:
manifest = json.loads((ROOT / "index" / "manifest.json").read_text(encoding="utf-8"))
print({k: manifest[k] for k in manifest if k in ("embedding_method", "dim", "documents", "chunks", "index_sha256", "counts")} or list(manifest))

{'counts': {'agencies': {'CDC': 9, 'NIH': 5, 'WHO': 8}, 'chunks': 79, 'documents': 22, 'superseded_documents': 2}}


## 2. The API: `POST /qa`

The real FastAPI app, called through its test client (same code path as `uvicorn`).

In [4]:
from fastapi.testclient import TestClient
from guiderail.api import create_app
client = TestClient(create_app(index_dir=ROOT / "index", threshold=0.6))
client.__enter__()
print(client.get("/health").json())

{'status': 'ok', 'version': '0.1.0', 'index_sha256': '45c3d9422161bc9750864850b8285b1feced930110530c2741ec0b9dd986cef8', 'documents': 22, 'chunks': 79, 'generator': 'extractive', 'threshold': 0.6}


In [5]:
def ask(q, top_k=5):
    body = client.post("/qa", json={"question": q, "top_k": top_k}).json()
    print("Q:", q); print("status:", body["status"], "| confidence:", body["confidence"])
    print("A:", body["answer"][:400]); 
    for c in body["citations"][:3]: print("  cite:", c["title"], "|", c["section"], "| v", c["version"], "| score", c["score"])
    print()
ask("How long must hands be washed with soap and water when visibly dirty?")
ask("What should be done with used needles?")

Q: How long must hands be washed with soap and water when visibly dirty?
status: validated | confidence: 0.875
A: Use soap and water when hands are visibly dirty, and after caring for a patient with Clostridioides difficile or norovirus, because alcohol is less effective against spores and non-enveloped viruses. Wash with soap and water for 40 to 60 seconds when hands are visibly soiled or contaminated with blood or body fluids. The duration for washing with soap and water increased from 30 seconds to 40 to 6
  cite: Hand Hygiene in Healthcare Settings | Recommendations | v 1 | score 0.356
  cite: Guidelines on Hand Hygiene in Health Care | Recommendations | v 2 | score 0.354
  cite: Guidelines on Hand Hygiene in Health Care | Changes from the 2019 edition | v 2 | score 0.192

Q: What should be done with used needles?
status: validated | confidence: 0.72
A: Do not recap used needles. If recapping is unavoidable, use a one-handed scoop technique.
  cite: Prevention and Management of Sha

**Look closely at the first answer.** It is validated, yet it also repeats the old 30-second figure from a changes note. This question is in the saved review list below as *validated but wrong*: a validated status is evidence-supported, not guaranteed correct.

### Two harder questions, shown as they came out

The first is not in the documents and is correctly sent to review (note that low-scoring citations are still listed so an analyst can look). The second is in scope, but the extractive generator returns scope sentences instead of the answer and the validator correctly refuses to validate it. Neither was chosen to look good.

In [6]:
ask("What is the recommended dose of amoxicillin for otitis media?")
ask("How long should a worker stay home after a respiratory illness?")

Q: What is the recommended dose of amoxicillin for otitis media?
status: needs_review | confidence: 0.323
A: This guidance sets out vaccines recommended for healthcare personnel and the practice steps for giving them. The recommended duration for hand rub increased from 15 to 20 seconds to 20 to 30 seconds. Give one dose of Tdap, then a Td or Tdap booster every 10 years.
  cite: Immunisation Practice and Adult Schedule for Healthcare Personnel | Scope | v 1 | score 0.08
  cite: Guidelines on Hand Hygiene in Health Care | Changes from the 2019 edition | v 2 | score 0.07
  cite: Immunisation Practice and Adult Schedule for Healthcare Personnel | Recommendations | v 1 | score 0.039

Q: How long should a worker stay home after a respiratory illness?
status: needs_review | confidence: 0.511
A: This guideline applies to all health workers who provide direct patient care in hospitals, clinics and long-term care facilities. It replaces the 2019 edition. This guidance applies to healthcare pers

### Input validation

In [7]:
for bad in ({"question": "", "top_k": 5}, {"question": "ok", "top_k": 0}, {"question": "ok", "top_k": 99}):
    print(bad, "->", client.post("/qa", json=bad).status_code)

{'question': '', 'top_k': 5} -> 422
{'question': 'ok', 'top_k': 0} -> 422
{'question': 'ok', 'top_k': 99} -> 422


## 3. Evaluation reports (generated from `evaluation/reports/`)

Retrieval Precision@K, mean confidence and pass rate at the threshold, for the held-out test split of the synthetic corpus.

In [8]:
from IPython.display import Markdown, display
display(Markdown((ROOT / "evaluation/reports/synthetic_test_extractive_cur_summary.md").read_text(encoding="utf-8")[:3200]))

# Evaluation summary: synthetic corpus, test split

- Index: `45c3d9422161bc97…`  |  generator: `extractive`  |  threshold: 0.6
- Queries: 20 in-corpus, 5 out-of-corpus

## Retrieval (in-corpus queries)

| Metric | Value |
|---|---|
| Precision@1 (document level) | 95.0% |
| Precision@5 (document level) | 65.0% |
| Precision@10 (document level) | 36.0% |
| Capped Precision@1 (vs. best attainable) | 95.0% |
| Capped Precision@5 (vs. best attainable) | 83.5% |
| Capped Precision@10 (vs. best attainable) | 83.9% |
| Hit@1 (a chunk containing the expected fact) | 70.0% |
| Hit@5 (a chunk containing the expected fact) | 100.0% |
| Hit@10 (a chunk containing the expected fact) | 100.0% |
| MRR (first fact-bearing chunk) | 0.8417 |

## Answers and validation

| Metric | Value |
|---|---|
| Pass rate (validated / in-corpus) | 90.0% |
| Mean confidence (in-corpus) | 0.8248 |
| Answer correct (expected facts present) | 85.0% |
| **Validated precision** (correct among validated) | 88.9% |
| **Validated but wrong** | 11.1% |
| Correct but flagged for review | 5.9% |
| Out-of-corpus sent to review | 100.0% |
| Out-of-corpus falsely validated | 0.0% |
| Decision accuracy (all queries) | 88.0% |

A high pass rate alone is not evidence of quality: an extractive answer is copied from the evidence, so coverage is near-perfect by construction. Validated precision and the validated-but-wrong rate are the meaningful numbers.

## Threshold sweep

| Threshold | Pass rate | Validated precision | Validated but wrong | Correct but flagged | OOC false accept | Decision accuracy |
|---|---|---|---|---|---|---|
| 0.3 | 100.0% | 85.0% | 15.0% | 0.0% | 60.0% | 76.0% |
| 0.35 | 100.0% | 85.0% | 15.0% | 0.0% | 40.0% | 80.0% |
| 0.4 | 100.0% | 85.0% | 15.0% | 0.0% | 40.0% | 80.0% |
| 0.45 | 100.0% | 85.0% | 15.0% | 0.0% | 40.0% | 80.0% |
| 0.5 | 95.0% | 84.2% | 15.8% | 5.9% | 40.0% | 76.0% |
| 0.55 | 90.0% | 88.9% | 11.1% | 5.9% | 40.0% | 80.0% |
| 0.6 | 90.0% | 88.9% | 11.1% | 5.9% | 0.0% | 88.0% |
| 0.65 | 90.0% | 88.9% | 11.1% | 5.9% | 0.0% | 88.0% |
| 0.7 | 90.0% | 88.9% | 11.1% | 5.9% | 0.0% | 88.0% |
| 0.75 | 85.0% | 88.2% | 11.8% | 11.8% | 0.0% | 84.0% |
| 0.8 | 80.0% | 87.5% | 12.5% | 17.6% | 0.0% | 80.0% |
| 0.85 | 45.0% | 88.9% | 11.1% | 52.9% | 0.0% | 60.0% |
| 0.9 | 20.0% | 100.0% | 0.0% | 76.5% | 0.0% | 48.0% |

## Items for analyst review (4)

- **S02** (VALIDATED BUT WRONG, confidence 0.875): How long must hands be washed with soap and water when visibly dirty?
- **S12** (flagged: low confidence, confidence 0.516): In which direction should cleaning proceed through a room?
- **S24** (VALIDATED BUT WRONG, confidence 0.814): What should be recorded in the notes after a shared decision?
- **S32** (correct answer withheld, confidence 0.473): Should used needles be put back in their caps?


### The same split with the Claude generator (saved run; not re-executed here)

In [9]:
display(Markdown((ROOT / "evaluation/reports/synthetic_test_llm_cur_summary.md").read_text(encoding="utf-8")[:3200]))

# Evaluation summary: synthetic corpus, test split

- Index: `45c3d9422161bc97…`  |  generator: `llm`  |  threshold: 0.6
- Queries: 20 in-corpus, 5 out-of-corpus

## Retrieval (in-corpus queries)

| Metric | Value |
|---|---|
| Precision@1 (document level) | 95.0% |
| Precision@5 (document level) | 65.0% |
| Precision@10 (document level) | 36.0% |
| Capped Precision@1 (vs. best attainable) | 95.0% |
| Capped Precision@5 (vs. best attainable) | 83.5% |
| Capped Precision@10 (vs. best attainable) | 83.9% |
| Hit@1 (a chunk containing the expected fact) | 70.0% |
| Hit@5 (a chunk containing the expected fact) | 100.0% |
| Hit@10 (a chunk containing the expected fact) | 100.0% |
| MRR (first fact-bearing chunk) | 0.8417 |

## Answers and validation

| Metric | Value |
|---|---|
| Pass rate (validated / in-corpus) | 70.0% |
| Mean confidence (in-corpus) | 0.7353 |
| Answer correct (expected facts present) | 90.0% |
| **Validated precision** (correct among validated) | 100.0% |
| **Validated but wrong** | 0.0% |
| Correct but flagged for review | 22.2% |
| Out-of-corpus sent to review | 100.0% |
| Out-of-corpus falsely validated | 0.0% |
| Decision accuracy (all queries) | 84.0% |

A high pass rate alone is not evidence of quality: an extractive answer is copied from the evidence, so coverage is near-perfect by construction. Validated precision and the validated-but-wrong rate are the meaningful numbers.

## Threshold sweep

| Threshold | Pass rate | Validated precision | Validated but wrong | Correct but flagged | OOC false accept | Decision accuracy |
|---|---|---|---|---|---|---|
| 0.3 | 100.0% | 90.0% | 10.0% | 0.0% | 0.0% | 92.0% |
| 0.35 | 100.0% | 90.0% | 10.0% | 0.0% | 0.0% | 92.0% |
| 0.4 | 85.0% | 100.0% | 0.0% | 5.6% | 0.0% | 96.0% |
| 0.45 | 80.0% | 100.0% | 0.0% | 11.1% | 0.0% | 92.0% |
| 0.5 | 75.0% | 100.0% | 0.0% | 16.7% | 0.0% | 88.0% |
| 0.55 | 75.0% | 100.0% | 0.0% | 16.7% | 0.0% | 88.0% |
| 0.6 | 70.0% | 100.0% | 0.0% | 22.2% | 0.0% | 84.0% |
| 0.65 | 70.0% | 100.0% | 0.0% | 22.2% | 0.0% | 84.0% |
| 0.7 | 70.0% | 100.0% | 0.0% | 22.2% | 0.0% | 84.0% |
| 0.75 | 70.0% | 100.0% | 0.0% | 22.2% | 0.0% | 84.0% |
| 0.8 | 55.0% | 100.0% | 0.0% | 38.9% | 0.0% | 72.0% |
| 0.85 | 35.0% | 100.0% | 0.0% | 61.1% | 0.0% | 56.0% |
| 0.9 | 20.0% | 100.0% | 0.0% | 77.8% | 0.0% | 44.0% |

## Items for analyst review (6)

- **S02** (flagged: low confidence, confidence 0.397): How long must hands be washed with soap and water when visibly dirty?
- **S04** (flagged: low confidence, confidence 0.375): Are artificial fingernails allowed for clinical staff?
- **S12** (correct answer withheld, confidence 0.428): In which direction should cleaning proceed through a room?
- **S18** (correct answer withheld, confidence 0.377): Do staff returning after illness still have to wear a mask?
- **S28** (correct answer withheld, confidence 0.569): How long should patients be observed after getting a vaccine?
- **S32** (correct answer withheld, confidence 0.473): Should used needles be put back in their caps?


## 4. Low-confidence queries for analyst review

The review list: validated-but-wrong answers, flagged answers and correct answers that were withheld.

In [10]:
import csv
rows = list(csv.DictReader(open(ROOT / "evaluation/reports/synthetic_test_extractive_cur_low_confidence.csv", encoding="utf-8")))
print(len(rows), "rows")
for r in rows[:8]: print(f'{r["id"]:>4} | {r["reason"]:<26} | conf {r["confidence"]} | {r["question"][:70]}')

4 rows
 S02 | VALIDATED BUT WRONG        | conf 0.875 | How long must hands be washed with soap and water when visibly dirty?
 S12 | flagged: low confidence    | conf 0.516 | In which direction should cleaning proceed through a room?
 S24 | VALIDATED BUT WRONG        | conf 0.814 | What should be recorded in the notes after a shared decision?
 S32 | correct answer withheld    | conf 0.473 | Should used needles be put back in their caps?


## 5. Real PDFs and the independent question sets

The table block below is read from the README, where it is generated from the saved reports. **The sealed sets are questions the system was never tuned on.** They are the better guide to real use.

In [11]:
text = (ROOT / "README.md").read_text(encoding="utf-8")
block = text.split("<!-- stats:start -->")[1].split("<!-- stats:end -->")[0]
display(Markdown(block))


_Generated by `scripts/check_docs_sync.py --write` from `evaluation/reports/`. Do not edit by hand._

- Synthetic corpus: 22 documents, 79 chunks (2 superseded versions).
- Real corpus: 5 PDFs from CDC and WHO, 1938 chunks.
- Index hash of the synthetic corpus at evaluation time: `45c3d9422161bc9750864850b8285b1feced930110530c2741ec0b9dd986cef8`.
- Tests collected: 214.
- Threshold 0.6; lexical re-ranker weight 0.5.

**Held-out test split, threshold 0.6** (95% Wilson interval in brackets; samples are small, so read [the full report](docs/EVALUATION_REPORT.md) before comparing rows):

| Corpus / generator | In-corpus n | Pass | Answer correct | Validated precision | Out-of-scope wrongly validated |
|---|---|---|---|---|---|
| synthetic / extractive | 20 | 90% (70–97) | 85% (64–95) | 89% (67–97) | 0 of 5 |
| synthetic / llm | 20 | 70% (48–85) | 90% (70–97) | 100% (78–100) | 0 of 5 |
| real / extractive | 24 | 96% (80–99) | 75% (55–88) | 78% (58–90) | 3 of 5 |
| real / llm | 24 | 75% (55–88) | 75% (55–88) | 94% (74–99) | 0 of 5 |

| Retrieval (test split) | Precision@5 | Capped Precision@5 | Hit@5 | MRR |
|---|---|---|---|---|
| synthetic | 65.0% | 83.5% | 100.0% | 0.842 |
| real | 92.5% | 92.5% | 83.3% | 0.711 |

**Sealed set (real PDFs; questions the system was never tuned on, run once):**

| Generator | In-corpus n | Pass | Answer correct | Validated precision | Out-of-scope wrongly validated |
|---|---|---|---|---|---|
| extractive | 16 | 94% (72–99) | 50% (28–72) | 47% (25–70) | 5 of 6 |
| llm | 16 | 62% (39–82) | 56% (33–77) | 80% (49–94) | 0 of 6 |

**Second sealed set (50 more of the owner's questions, same document; labels corrected once after the run, see the report):**

| Generator | In-corpus n | Pass | Answer correct | Validated precision | Out-of-scope wrongly validated |
|---|---|---|---|---|---|
| extractive | 49 | 86% (73–93) | 55% (41–68) | 55% (40–69) | 1 of 1 |
| llm | 49 | 61% (47–74) | 49% (36–63) | 70% (52–83) | 0 of 1 |

**Third sealed set, part 1 (CDC isolation guideline; all questions answerable, so no out-of-scope column):**

| Generator | In-corpus n | Pass | Answer correct | Validated precision |
|---|---|---|---|---|
| extractive | 47 | 83% (70–91) | 70% (56–81) | 67% (51–79) |
| llm | 47 | 66% (52–78) | 70% (56–81) | 84% (67–93) |

**Third sealed set, part 2 (CDC vaccine storage toolkit; all questions answerable, so no out-of-scope column):**

| Generator | In-corpus n | Pass | Answer correct | Validated precision |
|---|---|---|---|---|
| extractive | 47 | 89% (77–95) | 62% (47–74) | 69% (54–81) |
| llm | 47 | 83% (70–91) | 79% (65–88) | 92% (80–97) |

**Synthetic corpus v2 (documents of 7 to 9 chunks; fresh held-out questions; made after the v1 Precision@5 miss, so not independent):**

| Retrieval (v2 test) | Precision@1 | Precision@5 | Hit@5 | MRR |
|---|---|---|---|---|
| synthetic v2 | 95.0% | 59.0% | 100.0% | 0.917 |


## 6. What this does and does not show

- **Met:** deterministic indexing, the REST contract, the validator working independently of the generator, and honest reporting of every result.
- **Precision@5 of 0.65:** missed at first on the synthetic corpus; a document-cohesion re-rank (retrieval round 8) reached it on a fresh final set, judged under rules written first (report section 18). The brief's own v1 held-out figure sits right on the threshold.
- **Not met:** the Claude generator's 80% pass rate on the real-PDF held-out split.
- **Weak:** on real PDFs, retrieval is lexical and often misses the passage that answers a question worded differently from the document. The offline extractive generator can then validate a wrong answer; use the Claude generator for real documents. A semantic embedding model would help and was ruled out for this project.
- **Thin evidence:** the corpus and queries were written by the same AI that wrote the system; no clinician reviewed anything; only a few out-of-scope questions exist in the independent sets; Docker was verified on macOS only and the Windows setup never ran.

Full account: `docs/EVALUATION_REPORT.md` (sections 11 to 16), `docs/DEVELOPMENT_NOTES.md`.